# Salesforce NPSP Fundraising Q&A Demo

This notebook shows how to use **LlamaIndex** with donor data similar to what `SalesforceNPSPReader` produces. We use **synthetic data only** — no real Salesforce credentials required.

**For fundraisers**: This demo answers questions like "Who are our best prospects?" and "Which donors should we re-engage?" using AI over your donor summaries.

## Step 1: Create mock donor documents

These simulate the structured text the reader would return from Salesforce NPSP. Each document represents one donor with giving history and engagement data.

In [ ]:
from llama_index.core import Document

mock_docs = [
    Document(
        text="""Donor Profile: Jane Smith
Salesforce Contact ID: 003AAA
Primary Affiliation: City Hospital

Giving Summary:
  Total lifetime giving:   $125,000
  Number of gifts:         12
  Average gift amount:     $10,416
  Largest single gift:     $50,000
  First gift date:         2018-03-15
  Most recent gift date:   2024-01-20
  Soft credit total:       $5,000
  Planned giving count:    1

Engagement:
  Last CRM activity:       2024-02-01

Gift History (most recent 10):
  • $50,000 on 2024-01-20 | Type: Major Gift | Acknowledged
  • $10,000 on 2023-06-15 | Type: Annual | Acknowledged""",
        metadata={"donor_id": "003AAA", "donor_name": "Jane Smith", "total_gift_amount": 125000.0, "gift_count": 12, "last_gift_date": "2024-01-20", "last_activity_date": "2024-02-01", "affiliation": "City Hospital", "affinity_score": 92.5}
    ),
    Document(
        text="""Donor Profile: Robert Chen
Salesforce Contact ID: 003BBB
Primary Affiliation: Metro University

Giving Summary:
  Total lifetime giving:   $78,000
  Number of gifts:         8
  Average gift amount:     $9,750
  Largest single gift:     $30,000
  First gift date:         2019-11-01
  Most recent gift date:   2023-04-10
  Soft credit total:       $0
  Planned giving count:    0

Engagement:
  Last CRM activity:       2023-04-10

Gift History (most recent 10):
  • $15,000 on 2023-04-10 | Type: Major Gift | Acknowledged
  • $18,000 on 2022-12-01 | Type: Annual | Acknowledged""",
        metadata={"donor_id": "003BBB", "donor_name": "Robert Chen", "total_gift_amount": 78000.0, "gift_count": 8, "last_gift_date": "2023-04-10", "last_activity_date": "2023-04-10", "affiliation": "Metro University", "affinity_score": 85.0}
    ),
    Document(
        text="""Donor Profile: Maria Garcia
Salesforce Contact ID: 003CCC
Primary Affiliation: City Hospital

Giving Summary:
  Total lifetime giving:   $15,000
  Number of gifts:         4
  Average gift amount:     $3,750
  Largest single gift:     $8,000
  First gift date:         2021-06-15
  Most recent gift date:   2023-08-22
  Soft credit total:       $2,000
  Planned giving count:    0

Engagement:
  Last CRM activity:       2023-08-22

Gift History (most recent 10):
  • $5,000 on 2023-08-22 | Type: Annual | Acknowledged
  • $8,000 on 2022-09-01 | Type: Major Gift | Acknowledged""",
        metadata={"donor_id": "003CCC", "donor_name": "Maria Garcia", "total_gift_amount": 15000.0, "gift_count": 4, "last_gift_date": "2023-08-22", "last_activity_date": "2023-08-22", "affiliation": "City Hospital", "affinity_score": 72.0}
    ),
    Document(
        text="""Donor Profile: David Lee
Salesforce Contact ID: 003DDD
Primary Affiliation: Metro University

Giving Summary:
  Total lifetime giving:   $42,000
  Number of gifts:         6
  Average gift amount:     $7,000
  Largest single gift:     $20,000
  First gift date:         2020-02-01
  Most recent gift date:   2022-11-15
  Soft credit total:       $0
  Planned giving count:    1

Engagement:
  Last CRM activity:       2022-11-15

Gift History (most recent 10):
  • $20,000 on 2022-11-15 | Type: Major Gift | Acknowledged
  • $5,000 on 2021-10-01 | Type: Annual | Acknowledged""",
        metadata={"donor_id": "003DDD", "donor_name": "David Lee", "total_gift_amount": 42000.0, "gift_count": 6, "last_gift_date": "2022-11-15", "last_activity_date": "2022-11-15", "affiliation": "Metro University", "affinity_score": 68.0}
    ),
]

print(f"Created {len(mock_docs)} mock donor documents.")

## Step 2: Build the index and query engine

We create a VectorStoreIndex so we can search and query the donor data. Set `OPENAI_API_KEY` for real LLM responses, or use a local model.

In [ ]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_documents(mock_docs)
engine = index.as_query_engine()

print("Index built. You can now run the queries below.")

## Query 1: Which donors have given over $25,000 and haven't been contacted in 6 months?

In [ ]:
response1 = engine.query(
    "Which donors have given over $25,000 and haven't been contacted in 6 months?"
)
print(response1)

## Query 2: Who are the top prospects by lifetime giving affiliated with a hospital?

In [ ]:
response2 = engine.query(
    "Who are the top prospects by lifetime giving affiliated with a hospital?"
)
print(response2)

## Query 3: Which lapsed donors gave between $5,000 and $25,000 in the last 3 years?

In [ ]:
response3 = engine.query(
    "Which lapsed donors gave between $5,000 and $25,000 in the last 3 years?"
)
print(response3)

## Step 4: Using affinity_score_fn (mock scorer)

The SalesforceNPSPReader accepts an optional `affinity_score_fn` that injects an ML-derived propensity score into each document's metadata. Here's a mock scorer pattern — in production you'd use a trained model (e.g. PhilanthroPy):

In [ ]:
def mock_affinity_scorer(metadata: dict) -> float:
    """Simulate ML model: higher total_gift_amount + gift_count → higher score."""
    total = metadata.get("total_gift_amount", 0) or 0
    count = metadata.get("gift_count", 0) or 0
    base = min(50 + (total / 5000) + count * 2, 100)
    return round(base, 1)

# When using the real reader:
# reader = SalesforceNPSPReader(domain="login", affinity_score_fn=mock_affinity_scorer)
# docs = reader.load_data(limit=500)

print("Mock scorer: 125k total + 12 gifts →", mock_affinity_scorer({"total_gift_amount": 125000, "gift_count": 12}))
print("Mock scorer: 15k total + 4 gifts  →", mock_affinity_scorer({"total_gift_amount": 15000, "gift_count": 4}))